# 🔐 + 🚰 = 🔐🚰 Putting our API Key Management and Data Pipeline to Work

<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-One/blob/master/colab/Colab_FRED_Data.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-One/master?urlpath=lab/tree/notebooks/FRED_Data.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-One" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>
<br>

## 📖 What We Will Cover in this Notebook

After you understand the theory behind APIs and data ingestion, it is time to put our tools to the test. In this notebook, we will initialize our custom `financial_quant` package and securely connect to the **MASSIVE** data API.

More importantly, we are going to conquer the most frustrating part of quantitative research: **data management.**

**By the end of this notebook, you will be able to:**
1. **Securely vault API keys** so you never have to paste them into raw code again.
2. **Fetch granular futures data** (like the S&P 500 E-mini) using the `MASSIVEReader`.
3. **Leverage local caching** to speed up data requests by 300% and bypass API rate limits.
4. **Manage disk space** safely using built-in cache-clearing guardrails.

---

## 🛠️ Preparing the Notebook

<details>
<summary><b>👉 Click to Expand: 📦 Importing Libraries, Modules, and Functions</b></summary>

As a best practice, we always begin by importing our necessary dependencies in the very first code cell. Notice how lightweight our imports are here: just the `pandas`  library. ✨

This simplicity is a direct result of using the `financial_quant` package, which handles the heavy lifting behind the scenes. 🏗️ By keeping complicated setup details out of sight, we ensure the spotlight remains focused exactly where it belongs—on the core analysis. 🎯

```python
try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd
```
**👀 Keep an eye out**: As we progress, pay attention to how the `financial_quant` package is imported as `fq`, and how every reference to its functions begins with fq.. 💡 This follows the exact same standard practice we demonstrated in Chapter One with NumPy (np) and Pandas (pd).

</details>


In [ ]:
try:
    import pandas as pd
except:
    %pip -q install pandas
    import pandas as pd

## 📦 Getting Functions from financial_quant package

<details>
<summary><b style="font-size:1.2em; color: #1976d2; cursor: pointer;">🔌 Professional Packaging: How GitHub Installations Work</b></summary>
<br>
<p><b>The Logic:</b><br>
Usually, Python looks for modules as <code>.py</code> files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.</p>

<p><b>The Workflow:</b></p>
<ol>
<li><b>Fetch & Build:</b> The <code>%pip install git+https://...</code> command tells your Jupyter environment to clone the repository from GitHub and install the <code>financial_quant</code> package directly into your system's site-packages directory.</li>
<li><b>Import:</b> <code>import financial_quant as fq</code> loads the package into your notebook's memory and assigns it the quick alias <code>fq</code>.</li>
<li><b>Routing:</b> Behind the scenes, a special file called <code>__init__.py</code> ats as the package's "front door." It automatically gathers complex tools from deeply nested folders (like our fixed-income models and chart visualizers) and serves them up directly to the surface.</li>
<li><b>Execute:</b> You don't have to worry about where the files live. You just type <code>fq.one_y_axis() or fq.calc_ytm()</code>, and Python immediately knows where to route the request.</li>
</ol>

<p><b>Why do this?</b><br>
This is exactly how professional software engineering teams manage and distribute code. It keeps your notebooks incredibly clean, ensures everyone is using the exact same version of the math models, and guarantees your code is 100% portable to any cloud environment</p>
</details>

In [ ]:
# Use Python's built-in urllib to fetch the remote setup script
import urllib.request

# Download, decode, and execute the updater script directly from GitHub
updater_url = "https://raw.githubusercontent.com/PatrickJHess/quant_repo/master/fq_updater.py"
exec(urllib.request.urlopen(updater_url).read().decode('utf-8'))

# Run the installer/updater function and alias the package as 'fq'
fq = import_financial_quant()

☁️ Cloud environment detected. Installing fresh from GitHub...
✅ Installation complete!


<details>
<summary style="cursor: pointer;"><h3>🟢 Need a MASSIVE account or API key? <i>(Click to expand)</i></h3></summary>
<div style="margin-left: 20px; border-left: 3px solid #ccc; padding-left: 10px;">
  <ul>
    <li><a href="https://massive.com/dashboard/login" target="_blank"><b>Click here to go to the MASSIVE account page</b></a>.</li>
    <li>We highly recommend using the <b>"Sign in with Google"</b> option to instantly create your account and skip the email verification step.</li>
    <li><i>If you choose to use a different email, you must check your inbox for a verification link before proceeding.</i></li>
  </ul>
</div>
</details>

<br>

<details>
<summary style="cursor: pointer;"><h3>🔑 Access Your API key <i>(Click here after account setup.)</i></h3></summary>
<div style="margin-left: 20px; border-left: 3px solid #ccc; padding-left: 10px;">
  <ul>
    <li>Once you are logged in, <a href="https://massive.com/dashboard" target="_blank"><b>Shows Your API Key</b></a>.</li>
    <li>You can always access your key from this location</li>
  </ul>
</div>
</details>

<br>

<details>
<summary style="cursor: pointer;"><h3>📋 Copy your key <i>(Once MASSIVE displays your key, click here)</i></h3></summary>
<div style="margin-left: 20px; border-left: 3px solid #ccc; padding-left: 10px;">
  <ul>
    <li>Click the copy icon and the 32-character API key will be in your clickboard</li>
    <li><b>(Do not paste it directly into your code!)</b></li>
  </ul>
</div>
</details>

<br>

<details>
<summary style="cursor: pointer;"><h3>🚀 Initialize the Reader <i>(After copying the key, click here)</i></h3></summary>
<div style="margin-left: 20px; border-left: 3px solid #ccc; padding-left: 10px;">
  <p>Run the Python cell below to start the MASSIVE Reader. Because this is your first time, the notebook will realize you don't have a key yet and will actively help you set it up.</p>
  
  <ul>
    <li><b>If you are on Google Colab:</b> The notebook will display visual instructions guiding you to use Colab's built-in 🔑 <b>Secrets</b> sidebar to securely store your key.</li>
    <li><b>If you are running locally:</b> A secure text box will appear right beneath the cell for you to paste your key.</li>
  </ul>

  <p>Either way, once you paste the key you copied, the reader will vault it safely for all future notebooks!</p>
  
*Run this cell*. It will automatically prompt you for your key if it needs it.
<pre><code class="language-python">massive_data = fq.MASSIVEReader()</code></pre>

> **📁 Google Colab Users:** To use the cache, you will be prompted to grant this notebook access to your Google Drive. This allows the script to create a folder to save your downloaded CSVs.
</div>
</details>

<br>

<blockquote>
  <b>💡 Need to find your key again?</b><br>
  If you ever lose track of your key, don't worry! You can always view it by logging back in and visiting your <a href="https://massive.com/dashboard" target="_blank"><b>MASSIVE Dashboard</b></a>.
</blockquote>


In [ ]:
massive_data=fq.MASSIVEReader()

☁️ Colab environment detected. Attempting to mount Google Drive...
Mounted at /content/drive
📂 Cache anchored at: /content/drive/MyDrive/MASSIVE_DATA
✅ Key loaded seamlessly from Colab Secrets ('massive_key')


### 🔄 Two Readers. One Architecture.

MASSIVEReader and FredReader are built as perfect twins to keep your workflow seamless:

🚀 Same Syntax: Initialize both identically via fq.

📂 Same Caching: Predictable, isolated local storage.

🔐 Same Vault: Auto-loads API keys securely behind the scenes.

🧠 Learn one, master both!

In [ ]:
fred_data=fq.FredReader()

☁️ Colab environment detected. Attempting to mount Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Cache anchored at: /content/drive/MyDrive/FRED
✅ Key loaded seamlessly from Colab Secrets ('fred_key')


## ***🎬 Demonstrating the*** MASSIVEReader ***Class***

Once the MASSIVEReader instance (massive\_data) is initialized, you can use it to fetch market data.

While the reader includes several methods, this demonstration focuses on get\_futures\_data. 🎯 As the name implies, this method retrieves historical market data specifically for futures contracts. It requires three mandatory parameters and accepts two optional ones to customize your query. 🛠️

**📌 Required Arguments:**

* **🏷️ Contract Symbol:** A string representing the specific futures contract (e.g., 'ESU6' for the S\&P 500 contract expiring in September 2026).  
    
* **🟢 Start Date:** A string formatted as 'YYYY-MM-DD' (e.g., '2026-07-01' for July 1, 2026).  
    
* **🏁 End Date:** A string formatted as 'YYYY-MM-DD' (e.g., '2026-07-31' for July 31, 2026).  
  

**⚙️ Optional Arguments:**

* **⏱️** timespan**:** A string defining the time aggregation window. Accepted values are 'minute', 'hour', 'day', 'week', 'month', 'quarter', or 'year'. The default value is 'day'.  
   
* **🔢** multiplier**:** An integer used to scale the timespan. The default value is 1.  


*Note: These parameters map directly to the arguments built into the underlying Massive* RESTClient *used by the* MASSIVEReader*.*

### **🛡️ Navigating API Limitations**

Beyond just fetching data, MASSIVEReader works behind the scenes to seamlessly manage the underlying API's constraints so your code runs without interruption:

* **🚦 Rate Limits (Requests Per Minute):** The Freemium tier restricts users to five or fewer requests per minute.   

  * To prevent connection failures, the reader keeps track of the exact timestamp of each request. ⏱️     
  * If you approach the limit, it automatically "taps the brakes" and pauses execution until it is safe to proceed. 🛑  
   
* **📦 Data Volume Limits (Observations Per Request):** The API caps the amount of data returned in a single pull at 50,000 observations. 📊   

  * MASSIVEReader handles this by automatically caching data for all returned requests. 💾  
  * Using smart caching, the reader ensures that the next API request picks up exactly where the previous one left off. There is absolutely no need for you to manually account for new dates or handle pagination\! 🧠✨

## ⚡ The Power of Caching

The golden rule of quantitative research: **Never download the same data twice.**

When you request data using these readers, the system doesn't just hold it in active memory. It quietly writes a compressed copy directly to your hard drive (or your mounted Google Drive if you are using Colab).

**Let's run a speed test.**
In the cell below, we will download a large dataset for the first time. Pay attention to how long it takes. Then, run the exact same cell a second time. Because the data is now safely cached on your disk, the second load should be practically instantaneous, bypassing the API entirely.

Here is how we structure the request:
*   **`timespan='minute'`**: Requests minute-by-minute data.
*   **`multiplier=1`**: (The default) Leaves the timespan as 1-minute intervals.

```python
%%time
one_minute_data = massive_data.get_futures_data('ESU6', '2026-03-17', '2026-07-31', timespan='minute')
display(f'{one_minute_data.shape[0]} Rows In The DataFrame')
display(one_minute_data.tail())
```


### ☁️ No Cache- Get Fresh Data

In [ ]:
%%time
one_minute_data=massive_data.get_futures_data('ESU6','2026-03-17', '2026-07-31',timespan='minute')
display(f'{one_minute_data.shape[0]} Rows In The DataFrame')
display(one_minute_data.tail())

🚀 Fetching futures data for ESU6 (1 minute)...
☁️ Downloading ESU6 (1_minute) via API from 2026-03-17 to 2026-07-31...
⚠️ Max API limit of 50,000 rows reached. Run this cell again to continue fetching the rest of the date range!
💾 Merged and saved ESU6 to cache. (Total Coverage: 2026-03-16 to 2026-07-07)


'46992 Rows In The DataFrame'

,ticker,session_end_date,open,high,low,close,transactions,volume,dollar_volume
window_start,,,,,,,,,
2026-07-07 19:39:00,ESU6,2026-07-08,7545.50,7546.75,7545.00,7546.75,55,92,694237.25
2026-07-07 19:40:00,ESU6,2026-07-08,7546.50,7546.75,7544.00,7545.75,61,127,958251.25
2026-07-07 19:41:00,ESU6,2026-07-08,7545.75,7545.75,7545.25,7545.50,29,45,339549.25
2026-07-07 19:42:00,ESU6,2026-07-08,7545.50,7545.75,7545.00,7545.50,49,92,694167.00
2026-07-07 19:43:00,ESU6,2026-07-08,7545.50,7546.50,7544.00,7544.25,54,113,852589.50


CPU times: user 737 ms, sys: 13.7 ms, total: 750 ms
Wall time: 2.02 s


### 💡 Why did the download pause at 46,992 rows? 🛑

At 1-minute intervals, data adds up incredibly fast! The underlying API enforces a strict maximum of 50,000 rows per request.

MASSIVEReader tracks this volume in real-time. It safely halted the download on July 7th (at 46,992 rows) because pulling the entire next day of minute-by-minute data would have pushed the request over the 50k limit and caused an error. Instead, it securely caches the data it successfully pulled so you can simply re-run the cell to seamlessly fetch the next chunk! 🚀💾

### ⚡ Use Cache To Finish the Job

In [ ]:
%%time
one_minute_data=massive_data.get_futures_data('ESU6','2026-03-17', '2026-07-31',timespan='minute')
display(f'{one_minute_data.shape[0]} Rows In The DataFrame')
display(one_minute_data.tail())

🚀 Fetching futures data for ESU6 (1 minute)...
☁️ Downloading ESU6 (1_minute) via API from 2026-07-08 to 2026-07-31...
💾 Merged and saved ESU6 to cache. (Total Coverage: 2026-03-16 to 2026-07-31)


'71652 Rows In The DataFrame'

,ticker,session_end_date,open,high,low,close,transactions,volume,dollar_volume
window_start,,,,,,,,,
2026-07-31 16:55:00,ESU6,2026-07-31,7511.25,7511.50,7507.25,7508.25,367,813,6104606.50
2026-07-31 16:56:00,ESU6,2026-07-31,7508.00,7508.75,7504.00,7506.25,447,1177,8834635.75
2026-07-31 16:57:00,ESU6,2026-07-31,7506.50,7507.25,7502.75,7503.75,478,1494,11212100.25
2026-07-31 16:58:00,ESU6,2026-07-31,7503.75,7507.50,7503.25,7506.75,388,998,7490528.75
2026-07-31 16:59:00,ESU6,2026-07-31,7506.50,7507.00,7499.00,7503.25,769,2607,19560278.00


CPU times: user 887 ms, sys: 10.5 ms, total: 898 ms
Wall time: 1.68 s


### 🏁 Finishing the Job: Seamless Pagination in Action
By simply executing the exact same request a second time, MASSIVEReader perfectly demonstrates its smart caching and pagination abilities! Here is what just happened:

* 🧠 Smart Resume: The reader looked at the local cache and recognized that we already successfully downloaded data up to 2026-07-07.

* 🎯 Targeted API Call: Instead of starting from scratch and wasting API limits, it only requested the missing slice: 2026-07-08 through 2026-07-31.

* 🤝 Auto-Merge: It instantly stitched the newly downloaded chunk together with the previously cached chunk behind the scenes, updating the local cache to cover the full range.

* ✅ Mission Accomplished: We successfully bypassed the 50,000 row limit! As confirmed by display(one_minute_data.tail()), we made it all the way to our target end date with a complete, unbroken dataset of 71,652 rows.

### ⚡ Cache Saves Time And API Requests

In [ ]:
%%time
one_minute_data=massive_data.get_futures_data('ESU6','2026-03-17', '2026-07-31',timespan='minute')
display(f'{one_minute_data.shape[0]} Rows In The DataFrame')
display(one_minute_data.tail())

🚀 Fetching futures data for ESU6 (1 minute)...
⚡ Full cache hit for ESU6 (1_minute). Loaded directly from CSV.


'71652 Rows In The DataFrame'

,ticker,session_end_date,open,high,low,close,transactions,volume,dollar_volume
window_start,,,,,,,,,
2026-07-31 16:55:00,ESU6,2026-07-31,7511.25,7511.50,7507.25,7508.25,367,813,6104606.50
2026-07-31 16:56:00,ESU6,2026-07-31,7508.00,7508.75,7504.00,7506.25,447,1177,8834635.75
2026-07-31 16:57:00,ESU6,2026-07-31,7506.50,7507.25,7502.75,7503.75,478,1494,11212100.25
2026-07-31 16:58:00,ESU6,2026-07-31,7503.75,7507.50,7503.25,7506.75,388,998,7490528.75
2026-07-31 16:59:00,ESU6,2026-07-31,7506.50,7507.00,7499.00,7503.25,769,2607,19560278.00


CPU times: user 150 ms, sys: 3.99 ms, total: 154 ms
Wall time: 153 ms


💡 **What just happened:**

📈 This code gets minute-by-minute data for the futures contract symbol `ESU6` (the September 2026 E-mini S&P 500) between March 17, 2026, and July 31, 2026.

📊 The returned dataframe has more than 71,000 rows.

⏱️ **Fresh API Calls:** The two API calls takes 3.7 seconds to get the fresh data.

🏎️ **Cache Hit:** Acquiring the cached data is almost 25 times faster, taking only ~0.15 seconds!

## 🧹 When Cache Needs To Die

Cache is great as long as you are using it. But what happens when you're done? If you aren't using it, cache is just useless files cluttering your computer with the distinct disadvantage of becoming a major source of confusion.

Like `FredReader`, `MASSIVEReader` has a method built specifically to help you manage your disk space by purging old data: **`clear_cache()`**.

*(Note: Under the hood, this method calls the standalone `cache_clear()` utility function, passing its anchored cache directory automatically!)*

### 🛡️ How `clear_cache()` Works:
* 🎯 **Targeted Cleanup:** It accepts a `symbol` or series ID. Providing a partial symbol performs a case-sensitive search and returns an interactive list of potential matches to choose from.
* 💣 **Total Wipe:** If no symbol argument is provided, the method assumes you want to clear the entire directory and prompts you to delete all cached files at once.
* 🛑 **Safety First:** All delete requests require explicit confirmation (`'yes'`) before any file is touched, ensuring you never accidentally wipe out your hard-earned research data!

Let's test this guardrail by running the cell below to clear our temporary cache directory.

In [ ]:
massive_data.clear_cache('ESU6')

⚠️ Multiple files found for symbol 'ESU6':
  [1] ESU6_5_minute_data.csv
  [2] ESU6_1_day_data.csv
  [3] ESU6_1_minute_data.csv
  [all] Delete ALL of the above
  [0] Cancel

Enter file numbers to delete (e.g., '1', '1, 3'), 'all', or '0': 0
🚫 Operation cancelled.


### 💣 Clearing Everything Out

If you want to do a full reset and wipe the entire cache folder clean, simply omit all arguments when calling the method:

```python
massive_data.clear_cache()
```
#### What to expect:

1. The method will display a prominent 🚨🚨 double-warning showing the exact cache directory targeted for deletion.

2. It will pause and wait for you to type 'yes'.

3. If confirmed, it purges all stored files and subdirectories, leaving you with a clean slate!


In [ ]:
massive_data.clear_cache()

🚨🚨 WARNING: You are about to delete ALL cached data in:
   /content/drive/MyDrive/MASSIVE_DATA
Type 'yes' to proceed (press Enter or anything else to cancel): 
🛑 Operation cancelled. The cache was not modified.


## ⏱️ Granularity & Multipliers: Separate Files for Separate Frequencies

When you request data, `MASSIVEReader` builds a unique cache footprint using both the **`timespan`** and the **`multiplier`**.

*   **`timespan`**: Defines the unit of time (`'minute'`, `'hour'`, `'day'`, `'week'`, `'month'`, `'quarter'`, or `'year`).
*   **`multiplier`**: Sets the size of that unit (e.g., a `multiplier` of `5` with `timespan='minute'` creates 5-minute candles).

### ⚠️ Critical Rule: Different Frequencies = Isolated Caches
Because a 1-minute dataset is structurally different from a 5-minute or daily dataset, **they are never mixed or overwritten in the same cache file**.

If you request 1-minute data for `ESU6` and then request 5-minute data for `ESU6`, the reader intentionally creates two separate, isolated cache files on your disk:
1. `ESU6_1_minute.csv`
2. `ESU6_5_minute.csv`

Let's test this by fetching 5-minute bars for our S&P contract. Afterwards, we will call `clear_cache('ESU6')` to peek into our cache directory and see exactly how the reader organized these files:


In [ ]:
five_minute_data=massive_data.get_futures_data('ESU6','2026-03-17', '2026-07-01',timespan='minute',multiplier=5)
display(f'{five_minute_data.shape[0]} Rows In The DataFrame')
massive_data.clear_cache('ESU6')

🚀 Fetching futures data for ESU6 (5 minute)...
☁️ Downloading ESU6 (5_minute) via API from 2026-03-17 to 2026-04-30...
☁️ Downloading ESU6 (5_minute) via API from 2026-05-08 to 2026-07-01...
💾 Merged and saved ESU6 to cache. (Total Coverage: 2026-03-16 to 2026-06-30)


'13537 Rows In The DataFrame'

⚠️ Multiple files found for symbol 'ESU6':
  [1] ESU6_5_minute_data.csv
  [2] ESU6_1_day_data.csv
  [3] ESU6_1_minute_data.csv
  [all] Delete ALL of the above
  [0] Cancel

Enter file numbers to delete (e.g., '1', '1, 3'), 'all', or '0': 0
🚫 Operation cancelled.


💡 **The Evidence:**

Look closely at the interactive menu that just appeared! By searching for `ESU6`, the reader pulled up every single file associated with that ticker.

You can clearly see your 1-minute, 5-minute, and daily data all living side-by-side in perfect harmony. Requesting the 5-minute bars didn't destroy your 1-minute cache—the reader intelligently kept them isolated.

*(And by entering `0`, we safely cancelled the deletion, proving our guardrails work!)*

## 🎉 Conclusion: Infrastructure Conquered!

Before we write a single financial model, you have already solved the most frustrating part of quantitative research: **data management**. Let's review the toolkit you just mastered:

*   **Twin Architecture:** You can effortlessly pull both granular market data (`MASSIVEReader`) and macroeconomic indicators (`FredReader`) using the exact same syntax.
*   **Lightning Speed:** Your requests are instantly cached locally, dropping load times from seconds to milliseconds.
*   **Smart Appending:** The reader automatically stitches together missing date ranges, seamlessly bypassing API limits without duplicating data.
*   **Granular Isolation:** Your 1-minute, 5-minute, and daily datasets all live in perfect harmony without overwriting each other.
*   **Safety Guardrails:** You have strict, interactive controls to manage your disk space and prevent accidental data loss.

### ⏭️ What's Next?

Now that we are no longer fighting our data infrastructure, it is time to put these tools to work in the real world.

In the **next notebook**, we will dive into the actual mechanics of the futures market. We will use `MASSIVEReader` to explore contract rolling, the power of leverage, and the magic of margin.

Save your progress, close this notebook, and let's get into the markets! 📈